# Le but de ce notebook

L’objectif ici est de créer des fonctions réutilisables qui seront exploitées dans des pipelines de traitement de données.

Ces fonctions couvrent plusieurs étapes :

- Ouverture et lecture de fichiers (CSV / JSON)
- Nettoyage et vérification des données
- Séparation des lignes valides des anomalies

Ces fonctions doivent être le plus génériques possible, sans dépendre de valeurs codées en dur, afin d’être facilement réutilisables.

J’expérimente l’idée d’un pipeline "écrit en BDD", en schématisant les données plutôt qu’en utilisant de réelles bases de données, pour tester le concept.

## Partie 1 : Ouverture de fichiers

Les fonctions de cette partie servent à ouvrir un ou plusieurs fichiers CSV ou JSON dans le cadre d’une pipeline. Elles permettent :

- De localiser les fichiers dans les dossiers
- De détecter leur type (CSV / JSON)
- De les convertir en DataFrame avec pandas, pour les exploiter ensuite sans se soucier des extensions

In [45]:
import os
import re
import pandas as pd

DATA_ROOT = "../data"

### 1.1 Normalisation du chemin

Cette fonction permet de déterminer le dossier où se trouvent les fichiers à traiter.

- Elle se base sur une variable d’environnement comme DATA_ROOT
- On peut ensuite fournir en argument des sous-dossiers où l’on imagine stocker les fichiers CSV ou JSON

Cela garantit que les pipelines restent flexibles et portables.

In [46]:
# 1. Normalisation du chemin
def normalize_path(dossier_emplacement, data_root=DATA_ROOT):
    """
    Normalise le chemin du dossier relatif à ./data/
    """
    # Normaliser slashes
    dossier_emplacement = dossier_emplacement.replace("\\", "/").replace("//", "/")
    # Supprimer slash en début ou fin
    dossier_emplacement = dossier_emplacement.strip("/")
    # Construire chemin complet
    full_path = os.path.join(data_root, dossier_emplacement)
    # Normaliser le chemin final
    return os.path.normpath(full_path)

### 1.2 Création du motif Regex pour trouver le fichier

Cette fonction génère un motif regex pour identifier les fichiers à traiter.

- Arguments : nom_fixe, nom_variable, extension
    - nom_fixe : première partie du nom du fichier (ex. "Exercice")
    - nom_variable : partie variable du nom du fichier (ex. "YYYYMMDD")
    - extension : type de fichier (csv ou json)

Le motif généré permet de retrouver tous les fichiers correspondants, par exemple :

- CSV commençant par "Exercice"
- Avec 8 caractères pour la date
- Se terminant par .csv

In [47]:
# 2. Création du motif regex pour le fichier
def build_filename_pattern(nom_fixe, nom_variable, extension):
    if not extension.startswith("."):
        extension = "." + extension
    
    if nom_variable:
        # Le nom variable a une longueur connue : on remplace par autant de "."
        motif = f"^{re.escape(nom_fixe)}{'.' * len(nom_variable)}{re.escape(extension)}$"
    else:
        motif = f"^{re.escape(nom_fixe)}{re.escape(extension)}$"
    
    return motif

### 1.3 Recherche des fichiers correspondants

Cette fonction cherche les fichiers dans un dossier en fonction de la regex fournie :

- Arguments : dossier et motif regex
- Retour : liste des chemins de fichiers correspondants

⚠️ Actuellement, un print est utilisé si le dossier n’existe pas.

- En production, il serait préférable de remonter une erreur ou de logger l’information.

In [48]:
# 3. Recherche des fichiers correspondants et retourne les path
def find_matching_files(dossier, motif_regex):
    matched_files = []
    if os.path.exists(dossier) and os.path.isdir(dossier):
        for f in os.listdir(dossier):
            if re.match(motif_regex, f):
                matched_files.append(os.path.join(dossier, f))  # chemin complet
    else:
        print(f"[WARNING] Le dossier '{dossier}' n'existe pas.")
    return matched_files


# --- TESTS ---
dossier_emplacement = "\\raw\\"
nom_fichier_fixe = "daily_food_nutrition_dataset"
nom_fichier_variable = "YY"  # par exemple une date
extension_fichier = "csv"

# Normalisation du chemin
normalized_folder = normalize_path(dossier_emplacement)
print("Chemin normalisé :", normalized_folder)

# Construction du motif regex
pattern = build_filename_pattern(nom_fichier_fixe, nom_fichier_variable, extension_fichier)
print("Motif regex :", pattern)

# Recherche des fichiers
matched = find_matching_files(normalized_folder, pattern)
print("Fichiers trouvés :", matched)


# --- Autre exemple ---
dossier_emplacement2 = "raw/"
nom_fichier_fixe2 = "exercisedb_hobby"
nom_fichier_variable2 = ""
extension_fichier2 = "json"

normalized_folder2 = normalize_path(dossier_emplacement2)
pattern2 = build_filename_pattern(nom_fichier_fixe2, nom_fichier_variable2, extension_fichier2)
matched2 = find_matching_files(normalized_folder2, pattern2)

print("\nChemin normalisé 2 :", normalized_folder2)
print("Motif regex 2 :", pattern2)
print("Fichiers trouvés 2 :", matched2)

Chemin normalisé : ..\data\raw
Motif regex : ^daily_food_nutrition_dataset..\.csv$
Fichiers trouvés : ['..\\data\\raw\\daily_food_nutrition_dataset_1.csv', '..\\data\\raw\\daily_food_nutrition_dataset_2.csv']

Chemin normalisé 2 : ..\data\raw
Motif regex 2 : ^exercisedb_hobby\.json$
Fichiers trouvés 2 : ['..\\data\\raw\\exercisedb_hobby.json']


### 1.4 Ouverture et lecture des fichiers

Cette fonction prend un **chemin de fichier unique** et :

- Tente de le lire avec pandas selon son type (CSV ou JSON)
- Retourne un **DataFrame unique**
- Pour les CSV :
  - On considère que chaque fichier a une ligne header
  - Les lignes qui ne correspondent pas au nombre de colonnes du header sont automatiquement ignorées

⚠️ Actuellement, un print s’affiche si :

- Ce n’est ni un CSV ni un JSON
- Le fichier n’existe pas ou ne peut pas être lu
- En production, ces prints devraient être remplacés par des logs ou exceptions

💡 Notes complémentaires :

- Une seconde fonction `read_files` permet de traiter une **liste de fichiers**, mais elle utilise simplement `read_single_file` en boucle. Elle est moins utile si l’on souhaite suivre individuellement chaque fichier (logs, comptage des lignes ignorées, etc.)
- On pourrait également comptabiliser le nombre de lignes supprimées dans les CSV pour un suivi et un logging plus précis

In [49]:
# Lecture automatique avec pandas
def read_single_file_with_pandas(file_path):
    """
    Lit un fichier CSV ou JSON et retourne un DataFrame.
    Retourne None si l'extension n'est pas supportée ou en cas d'erreur.
    """
    ext = os.path.splitext(file_path)[1].lower()
    try:
        if ext == ".csv":
            df = pd.read_csv(file_path, header=0, on_bad_lines='skip') # header obligatoire et saute les lignes incorrectes (nb colonne different du header)
        elif ext == ".json":
            df = pd.read_json(file_path)
        else:
            print(f"[WARNING] Extension non supportée pour {file_path}, ignoré.")
            return None
        return df
    except Exception as e:
        print(f"[ERROR] Impossible de lire {file_path}: {e}")
        return None


def read_files_with_pandas(file_paths):
    """
    Lit une liste de fichiers CSV ou JSON et retourne une liste de DataFrames
    en utilisant read_single_file_with_pandas.
    """
    dfs = []
    for fpath in file_paths:
        df = read_single_file_with_pandas(fpath)
        if df is not None:
            dfs.append(df)
    return dfs


# ---TEST---
dfs = read_files_with_pandas(matched)
print(f"Nombre de DataFrames lus : {len(dfs)}")
if dfs:
    print(dfs[0].head())  # afficher les 5 premières lignes du premier fichier

Nombre de DataFrames lus : 2
                     Food_Item       Category  Calories (kcal)  Protein (g)  \
0     Scrambled Eggs (2 large)  Protein/Dairy              180         12.0   
1  Whole Wheat Toast (1 slice)          Grain               80          4.0   
2               Coffee (black)       Beverage                5          0.3   
3                       Banana          Fruit              105          1.3   
4        Grilled Chicken Salad   Meal/Protein              350         30.0   

   Carbohydrates (g)  Fat (g)  Fiber (g)  Sugars (g)  Sodium (mg)  \
0                2.0     14.0        0.0         1.0          180   
1               14.0      1.0        2.0         2.0          140   
2                0.0      0.1        0.0         0.0            5   
3               27.0      0.4        3.1        14.0            1   
4               10.0     20.0        5.0         4.0          400   

   Cholesterol (mg)  Meal_Type  Water_Intake (ml)  
0               370  Breakfas